#Build Races Fact table
1. Read silver 'races' table
2. Read silver 'sprints' table
3. Add a new column session_type with values RACE or SPRINT
4. UNION 'results' and 'sprints'
5. Derive additional columns
    - is_win
    - is_podium
    - has_points
6. Write the table to the gold layer

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

###Step 1. Read the silver tables

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
        .filter(F.col("batch_id") == v_batch_id)
        .withColumn("session_type", F.lit("RACE"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
        .filter(F.col("batch_id") == v_batch_id)
        .withColumn("session_type", F.lit("SPRINT"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file", "batch_id", "created_timestamp", "updated_timestamp")
)

###Step 2. UNION the tables

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

###Step 3. Add derived columns

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1,3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season == 2025"))

round,season,constructor_id,driver_id,grid_position,completed_laps,car_number,points,finish_position,finish_position_text,status,session_type,is_win,is_podium,has_points
1,2025,alpine,doohan,14,0,7,0.0,19,R,Retired,RACE,false,false,false
1,2025,alpine,gasly,9,57,10,0.0,11,11,Finished,RACE,false,false,false
1,2025,aston_martin,alonso,12,32,14,0.0,17,R,Retired,RACE,false,false,false
1,2025,aston_martin,stroll,13,57,18,8.0,6,6,Finished,RACE,false,false,true
1,2025,ferrari,hamilton,8,57,44,1.0,10,10,Finished,RACE,false,false,true
1,2025,ferrari,leclerc,7,57,16,4.0,8,8,Finished,RACE,false,false,true
1,2025,haas,bearman,20,57,87,0.0,14,14,Finished,RACE,false,false,false
1,2025,haas,ocon,19,57,31,0.0,13,13,Finished,RACE,false,false,false
1,2025,mclaren,norris,1,57,4,25.0,1,1,Finished,RACE,true,true,true
1,2025,mclaren,piastri,2,57,81,2.0,9,9,Finished,RACE,false,false,true


###Step 6. Write the data

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    target_table=target_table,
    merge_condition="""
        t.season = s.season
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
    """,
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finish_position",
        "finish_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]
)